# 02 — Indexação no Elasticsearch e Validação

Indexa o catálogo processado em um índice BM25 e em um índice KNN por modelo de embedding, e valida o resultado.
Este notebook substitui o antigo `scripts/03_index_all_models.py`.

**Pré-requisito:** `data/processed/products_us.parquet` (gerado pelo notebook 01).

**Atenção (Colab):** o Elasticsearch iniciado aqui roda na VM e seus índices são perdidos quando a sessão termina.
Execute o notebook 03 (avaliação) na mesma sessão, ou aponte `ES_URL` para um servidor externo.

Para alternar CPU/GPU, altere apenas `DEVICE` na célula de setup.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# ── Configuração única ───────────────────────────────────────────────
DEVICE = "auto"                     # "cpu" | "cuda" | "auto"
DRIVE_PROJECT_DIR = "MyDrive/tcc"   # pasta do projeto no Google Drive (só Colab)
# ─────────────────────────────────────────────────────────────────────

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive") / DRIVE_PROJECT_DIR
    %pip install -q polars pyarrow matplotlib elasticsearch~=8.15 sentence-transformers tqdm
else:
    PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / "src").exists(), f"Pasta 'src' não encontrada em {PROJECT_ROOT}"
sys.path.insert(0, str(PROJECT_ROOT))

from src.device import build_runtime_config, collect

RUNTIME = build_runtime_config(DEVICE)
print(f"Projeto : {PROJECT_ROOT}")
print(f"Colab   : {IN_COLAB}")
print(f"Device  : {RUNTIME.device} (batch_size={RUNTIME.batch_size}, max_seq_length={RUNTIME.max_seq_length})")

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from elasticsearch import Elasticsearch

from src.elasticsearch.indexer import Indexer, prepare_dataframe
from src.elasticsearch.searcher import Searcher, SearchStrategy
from src.models.core import unload_all_models
from src.models.registry import BM25_INDEX_NAME, list_models

PRODUCTS_PATH = PROJECT_ROOT / "data" / "processed" / "products_us.parquet"
REPORT_DIR = PROJECT_ROOT / "results" / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
assert PRODUCTS_PATH.exists(), f"{PRODUCTS_PATH} não encontrado. Execute o notebook 01 primeiro."

# ── Parâmetros de indexação ──────────────────────────────────────────
MODELS: list[str] | None = None   # None = todos; ex.: ["bge", "rexbert"]
SKIP_BM25 = False
OVERWRITE = False                 # True apaga e recria índices existentes
BULK_CHUNK_SIZE = 256
BATCH_SIZE = RUNTIME.batch_size
MAX_SEQ_LENGTH = RUNTIME.max_seq_length

# ── Elasticsearch ────────────────────────────────────────────────────
START_LOCAL_ES = IN_COLAB         # no Colab, baixa e inicia um ES na VM
ES_VERSION = "8.15.0"
ES_HEAP = "2g"
ES_URL = "http://localhost:9200"
ES_USER = "elastic"
ES_PASSWORD = os.environ.get("ES_PASSWORD")   # None = sem autenticação

## 1. Elasticsearch

No Colab o servidor é iniciado com segurança desativada (acesso apenas local à VM). Fora do Colab, suba o
Elasticsearch por conta própria (por exemplo via Docker) e ajuste `ES_URL`, `ES_USER` e a variável `ES_PASSWORD`.

In [ ]:
def start_local_elasticsearch(version: str, heap: str) -> subprocess.Popen:
    """Baixa, extrai e inicia um Elasticsearch single-node (Linux, executa como usuário 'daemon')."""
    base = Path("/content")
    home = base / f"elasticsearch-{version}"
    if not home.exists():
        tarball = base / f"{home.name}-linux-x86_64.tar.gz"
        url = f"https://artifacts.elastic.co/downloads/elasticsearch/{tarball.name}"
        subprocess.run(["wget", "-q", url, "-O", str(tarball)], check=True)
        subprocess.run(["tar", "-xzf", str(tarball), "-C", str(base)], check=True)
    subprocess.run(["chown", "-R", "daemon:daemon", str(home)], check=True)
    return subprocess.Popen(
        [
            "sudo", "-E", "-u", "daemon", str(home / "bin" / "elasticsearch"),
            "-E", "xpack.security.enabled=false",
            "-E", "discovery.type=single-node",
        ],
        env={**os.environ, "ES_JAVA_OPTS": f"-Xms{heap} -Xmx{heap}"},
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )


if START_LOCAL_ES:
    es_process = start_local_elasticsearch(ES_VERSION, ES_HEAP)

client_kwargs: dict = {"request_timeout": 60}
if ES_PASSWORD:
    client_kwargs |= {"basic_auth": (ES_USER, ES_PASSWORD), "verify_certs": False}
es = Elasticsearch(ES_URL, **client_kwargs)

for attempt in range(60):
    try:
        info = es.info()
        break
    except Exception:
        time.sleep(3)
else:
    raise RuntimeError(f"Elasticsearch não respondeu em {ES_URL}")

print(f"Elasticsearch {info['version']['number']} conectado em {ES_URL}")

A busca híbrida (RRF) exige licença que o plano gratuito não inclui. A célula abaixo ativa a licença trial de 30 dias
no servidor local; em um servidor externo sem a licença, a estratégia `hybrid` falhará no notebook 03.

In [ ]:
try:
    es.license.post_start_trial(acknowledge=True)
except Exception as exc:
    print(f"Não foi possível iniciar a licença trial: {exc}")
print("Licença:", es.license.get()["license"]["type"])

## 2. Preparação do catálogo

In [ ]:
products = collect(pl.scan_parquet(PRODUCTS_PATH), RUNTIME.device)
prepared = prepare_dataframe(products)
print(f"Documentos preparados: {prepared.height:,}")
prepared.select("product_id", "product_title", "product_text").head(3)

## 3. Indexação

In [ ]:
def format_elapsed(seconds: float) -> str:
    minutes, secs = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours:02d}h{minutes:02d}m{secs:02d}s" if hours else f"{minutes:02d}m{secs:02d}s"


indexer = Indexer(
    es=es,
    device=RUNTIME.device,
    batch_size=BATCH_SIZE,
    bulk_chunk_size=BULK_CHUNK_SIZE,
    overwrite=OVERWRITE,
)

targets = MODELS or [cfg.alias for cfg in list_models()]
summary: list[dict] = []
total_start = time.perf_counter()

if not SKIP_BM25:
    t0 = time.perf_counter()
    indexer.index_bm25(prepared)
    summary.append({"index": "bm25", "status": "ok", "elapsed": format_elapsed(time.perf_counter() - t0)})

for alias in targets:
    t0 = time.perf_counter()
    try:
        indexer.index_model(alias, prepared, max_seq_length_override=MAX_SEQ_LENGTH)
        status, error = "ok", ""
    except Exception as exc:
        status, error = "error", str(exc)
    finally:
        unload_all_models()
    summary.append({"index": alias, "status": status, "error": error, "elapsed": format_elapsed(time.perf_counter() - t0)})

print(f"Tempo total: {format_elapsed(time.perf_counter() - total_start)}")
pl.DataFrame(summary, infer_schema_length=None)

## 4. Inventário dos índices

In [ ]:
expected_docs = prepared.height
index_names = [cfg.es_index_name for cfg in list_models()] + [BM25_INDEX_NAME]

rows = []
for index_name in index_names:
    if not es.indices.exists(index=index_name):
        rows.append({"index": index_name, "docs": None, "size_gb": None, "ok": False})
        continue
    stats = es.indices.stats(index=index_name)["_all"]["primaries"]
    docs = stats["docs"]["count"]
    rows.append({
        "index": index_name,
        "docs": docs,
        "size_gb": round(stats["store"]["size_in_bytes"] / 1024**3, 2),
        "ok": docs == expected_docs,
    })

inventory = pl.DataFrame(rows, schema={"index": pl.String, "docs": pl.Int64, "size_gb": pl.Float64, "ok": pl.Boolean})
print(f"Documentos esperados por índice: {expected_docs:,}")
inventory

In [ ]:
existing = inventory.drop_nulls("size_gb")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(existing["index"].to_list(), existing["size_gb"].to_list(), color="#1f77b4")
ax.set_xlabel("Tamanho em disco (GB)")
ax.set_title("Tamanho dos índices")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT_DIR / "indexing_sizes.png", bbox_inches="tight")
plt.show()

## 5. Sanidade dos embeddings

In [ ]:
N_SAMPLE = 50

norm_rows = []
for cfg in list_models():
    if not es.indices.exists(index=cfg.es_index_name):
        continue
    response = es.search(
        index=cfg.es_index_name,
        size=N_SAMPLE,
        query={"function_score": {"random_score": {}}},
        source=["embedding"],
    )
    vectors = [hit["_source"]["embedding"] for hit in response["hits"]["hits"] if "embedding" in hit["_source"]]
    if not vectors:
        print(f"{cfg.alias}: nenhum embedding retornado em _source")
        continue
    norms = pl.Series([sum(v * v for v in vec) ** 0.5 for vec in vectors])
    norm_rows.append({
        "model": cfg.alias,
        "dim_ok": all(len(vec) == cfg.embedding_dim for vec in vectors),
        "norm_mean": norms.mean(),
        "norm_min": norms.min(),
        "norm_max": norms.max(),
    })

print("Normas L2 esperadas: ~1.0 (normalize_embeddings=True)")
pl.DataFrame(norm_rows).with_columns(pl.col("^norm_.*$").round(6))

## 6. Buscas manuais

In [ ]:
SAMPLE_MODEL = "bge"
SAMPLE_QUERIES = ["wireless noise cancelling headphones", "running shoes for women", "usb c cable 6ft"]

searcher = Searcher(es, device=RUNTIME.device)

for query in SAMPLE_QUERIES:
    print(f"\n=== {query} ===")
    for strategy in SearchStrategy:
        try:
            hits = searcher.search(SAMPLE_MODEL, query, strategy=strategy, k=3)
        except Exception as exc:
            print(f"[{strategy.value}] erro: {exc}")
            continue
        print(f"[{strategy.value}]")
        for hit in hits:
            print(f"  {hit.rank}. {hit.score:.3f}  {hit.product_title[:90]}")

searcher.unload_model(SAMPLE_MODEL)

## 7. Comparação entre modelos para uma mesma query

Compara o top-K da busca KNN em cada índice de embedding.

In [ ]:
COMPARISON_QUERY = "wireless bluetooth headphones"
K_COMPARE = 5

print(f"Query: '{COMPARISON_QUERY}' — top {K_COMPARE} por modelo (KNN)")
for cfg in list_models():
    if not es.indices.exists(index=cfg.es_index_name):
        print(f"\n[{cfg.alias}] índice não encontrado, pulando.")
        continue
    hits = searcher.search(cfg.alias, COMPARISON_QUERY, strategy=SearchStrategy.KNN, k=K_COMPARE)
    print(f"\n[{cfg.display_name}]")
    for hit in hits:
        print(f"  #{hit.rank}  score={hit.score:.4f}  {hit.product_title[:70]}")
    searcher.unload_model(cfg.alias)

## 8. Checklist final

In [ ]:
print("CHECKLIST DE VALIDAÇÃO DA INDEXAÇÃO")
print("=" * 55)
for row in inventory.iter_rows(named=True):
    icon = "OK  " if row["ok"] else "FALHA"
    size = f"{row['size_gb']:.2f} GB" if row["size_gb"] is not None else "N/A"
    docs = f"{row['docs']:,}" if row["docs"] is not None else "N/A"
    print(f"  [{icon}] {row['index']:<25} docs={docs:<10} tamanho={size}")

if inventory["ok"].all():
    print("\nTodos os índices estão corretos. Prossiga para o notebook 03 (avaliação).")
else:
    print("\nHá índices ausentes ou incompletos. Ajuste MODELS/OVERWRITE e reexecute a seção 3.")